In [14]:
# A default setup cell.
# It imports environment variables, define 'devtools.debug" as a buildins, set PYTHONPATH, and code auto-reload
# Copy it in other Notebooks


from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)


%load_ext autoreload
%autoreload 2
%reset -f

In [15]:
from src.utils.collection_helpers import find_by_key
from src.utils.config_mngr import global_config, global_config_reload

global_config_reload()

list_demos = global_config().merge_with("config/demos/document_extractor.yaml").get_list("Document_extractor_demo")

d = find_by_key(list_demos, "schema_name", "test")
assert d

AssertionError: 

In [ ]:
[item for item in list_demos if item.schema_name == "test" ]

AttributeError: 'dict' object has no attribute 'schema_name'

In [ ]:
next((item for item in list_demos if getattr(item, "schema_name") == "test"), None)

AttributeError: 'dict' object has no attribute 'schema_name'

In [16]:
d

In [17]:
list_demos

[{'schema_name': 'test',
  'key': 'Person.name',
  'top_class': 'Person',
  'schema': {'Person': {'description': 'class for members of the family',
    'fields': {'name': {'description': "Person's full name", 'required': True},
     'age': {'type': 'int', 'description': 'Age in years', 'required': False},
     'email': {'type': 'list[Email]',
      'description': 'Email addresses',
      'required': True},
     'address': {'type': 'Address',
      'description': 'Home address',
      'required': False}}},
   'Email': {'description': 'email contact',
    'fields': {'url': {'type': 'str', 'required': True},
     'email_type': {'type': 'str', 'required': False}}},
   'Address': {'description': 'postal contact',
    'fields': {'street': {'type': 'str', 'required': True},
     'city': {'type': 'str', 'required': True},
     'zip_code': {'type': 'str', 'required': False},
     'country': {'type': 'str', 'required': False}}}}},
 {'schema_name': 'Rainbow File',
  'version': 1.0,
  'key': 'Proj

In [ ]:
EXAMPLE_YAML = """
Person:
  description: Basic contact card
  fields:
    name:
      type: str
      description: Full name of the person
    age:
      type: int
      description: Age in years
    email:
      type: str
      description: Primary e-mail address
"""

url = global_config().get_dsn("vector_store.postgres_url", driver="asyncpg")
print(url)

rag = PydanticRag(
    model_definition=EXAMPLE_YAML,
    postgres_url=url,
    embeddings_id="qwen3_06b_deepinfra",
    llm_id=None,
)

# A tiny markdown document
doc_text = """
# Jane Doe
Jane is 29 years old and can be reached at jane.doe@example.com.
"""

# 1. Analyse → structured Pydantic object
person = rag.analyze_document(doc_text)
print("Structured result:", person)

# 2. Index the document
rag.store_document(person)
print("Document stored.")

# 3. Query the vector store
hits = rag.query_vectorstore("e-mail address", k=2)
print("Vector hits:", hits)